Import Librerie

In [2]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

Caricamento e preparazione dei dati

In [3]:
# Carichiamo i file
df_train = pd.read_csv("train_set copia.csv")
df_test = pd.read_csv("test_set copia.csv")

# Codifichiamo le etichette
le = LabelEncoder()
df_train['label'] = le.fit_transform(df_train['label'])
df_test['label'] = le.transform(df_test['label'])

print(f"Classi trovate: {le.classes_}")

# Creiamo il formato per BERT
train_ds = Dataset.from_pandas(df_train)
test_ds = Dataset.from_pandas(df_test)
raw_datasets = DatasetDict({"train": train_ds, "test": test_ds})

Classi trovate: ['bug' 'feature']


Tokenizzazione

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    # Usiamo la colonna 'text'
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/32650 [00:00<?, ? examples/s]

Map:   0%|          | 0/13994 [00:00<?, ? examples/s]

Configurazione modello e metriche

In [5]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Congeliamo i pesi di BERT per fare solo 'Feature Extraction'
for param in model.bert.parameters():
    param.requires_grad = False

def compute_metrics(eval_pred):
    metric_acc = evaluate.load("accuracy")
    metric_f1 = evaluate.load("f1")
    metric_precision = evaluate.load("precision")
    metric_recall = evaluate.load("recall")

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": metric_acc.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": metric_f1.compute(predictions=predictions, references=labels)["f1"],
        "precision": metric_precision.compute(predictions=predictions, references=labels)["precision"],
        "recall": metric_recall.compute(predictions=predictions, references=labels)["recall"],
    }

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Addestramento

In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True # Fondamentale per la velocità su GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.678644,0.668589,0.599114,0.656839,0.574163,0.767329
2,0.669243,0.664732,0.588466,0.665854,0.560461,0.820066
3,0.665863,0.662197,0.597042,0.666312,0.568572,0.804631


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3063, training_loss=0.673773126356204, metrics={'train_runtime': 329.2239, 'train_samples_per_second': 297.518, 'train_steps_per_second': 9.304, 'total_flos': 6442931968128000.0, 'train_loss': 0.673773126356204, 'epoch': 3.0})

Notiamo che i valori delle matriche non sono ottimali quindi sperimentiamo non congelando i neuroni

In [8]:
# Carichiamo una nuova istanza del modello per il Fine-Tuning completo
model_ft = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# In questo modo tutti i 110 milioni di parametri di BERT verranno addestrati.
print("Modello sbloccato pronto per il Fine-Tuning completo.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modello sbloccato pronto per il Fine-Tuning completo.


Training Argument per fine-tuning completo

In [9]:
training_args_ft = TrainingArguments(
    output_dir="./results_fine_tuning",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,               # LR classica per fine-tuning BERT
    per_device_train_batch_size=16,   # Abbassiamo a 16 per evitare errori di memoria (Full FT consuma più RAM)
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True
)

Nuovo trainer e avvio

In [10]:
trainer_ft = Trainer(
    model=model_ft,
    args=training_args_ft,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# Avvia il training pesante
print("Inizio Fine-Tuning completo (durata stimata: 25-35 minuti)...")
trainer_ft.train()

Inizio Fine-Tuning completo (durata stimata: 25-35 minuti)...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.259323,0.279633,0.900815,0.900559,0.902888,0.898242
2,0.210147,0.272539,0.905960,0.903079,0.931621,0.876233
3,0.162725,0.314365,0.903387,0.902915,0.907346,0.898528


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=6123, training_loss=0.22333988760998488, metrics={'train_runtime': 980.7413, 'train_samples_per_second': 99.873, 'train_steps_per_second': 6.243, 'total_flos': 6442931968128000.0, 'train_loss': 0.22333988760998488, 'epoch': 3.0})

Il miglioramento è notevole, lo riteniamo opportuno per decidere di scaricare questo modello

In [11]:
# Salva il modello migliore
trainer_ft.save_model("./github_bert_finetuned")

# Crea lo zip specifico per questo modello
!zip -r github_model_FINETUNED.zip ./github_bert_finetuned

print("Fine-Tuning completato! Scarica 'github_model_FINETUNED.zip' dalla barra a sinistra.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: github_bert_finetuned/ (stored 0%)
  adding: github_bert_finetuned/tokenizer.json (deflated 71%)
  adding: github_bert_finetuned/training_args.bin (deflated 53%)
  adding: github_bert_finetuned/model.safetensors (deflated 7%)
  adding: github_bert_finetuned/config.json (deflated 53%)
  adding: github_bert_finetuned/tokenizer_config.json (deflated 42%)
Fine-Tuning completato! Scarica 'github_model_FINETUNED.zip' dalla barra a sinistra.


Il procedimento di fine-tuning è stato fatto mediante googlecolab. 

Questo è il notebook scaricato.